# Import and inspect data

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize, FreqDist
import string

In [6]:
df = pd.read_csv('../data/complaints_processed_full.csv')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143962 entries, 0 to 143961
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   product    143962 non-null  object
 1   narrative  143962 non-null  object
dtypes: object(2)
memory usage: 2.2+ MB


In [8]:
df.head()

,product,narrative
0,credit_reporting,account
1,credit_reporting,wrote three request unverified account listed ...
2,credit_reporting,recently going check new car carefully conside...
3,mortgages_and_loans,call hour weekend using various number
4,credit_reporting,notified experian inaccuracy report national c...


### Isolate relevant columns

In [5]:
df = df[['Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative']]

KeyError: "None of [Index(['Product', 'Sub-product', 'Issue', 'Sub-issue',\n       'Consumer complaint narrative'],\n      dtype='object')] are in the [columns]"

In [ ]:
df = df.rename(columns={"Product": "product", "Sub-product": "subproduct", "Issue": "issue", "Sub-issue": "subissue", "Consumer complaint narrative": "narrative"})

In [ ]:
df['product'].value_counts()

In [ ]:
df['subproduct'].value_counts()

In [ ]:
df['issue'].value_counts()

In [ ]:
df['subissue'].value_counts()

Except for 'product', it doesn't seem there's enough data to train on. Maybe I could do subproduct, too, but I'd want to eliminate those with the lower value counts.

Note for subproduct, a high number is "I don't know." I could maybe classify those. 

In 'subissue', the second highest is 'None'. Also, there seems to be a lot of overlap between the categories, finer points that some consumers may not make. Example: "Debt is not yours" vs. "Debt was result of identity theft".

# Prepare Text

## Inspect first row

In [ ]:
df['narrative'].head(5)

In [ ]:
print(df.iloc[0])
text = df['narrative'][2]
text

## Process Data

### Function to tokenize data and remove stopwords

In [ ]:
stopwords_list = stopwords.words('english') + list(string.punctuation)
stopwords_list += ["''", '""', '...', '``']

In [ ]:
string.punctuation

In [ ]:
list(string.punctuation)

In [ ]:
list('abc')

In [ ]:
def process_narrative(narrative):
    tokens = nltk.word_tokenize(narrative)
    stopwords_removed = [token.lower() for token in tokens if token.lower() not in stopwords_list]
    return stopwords_removed  

### Inspect and process first narrative 

In [ ]:
text_words = process_narrative(text)
text_words[0:10]

In [ ]:
# Add to stopwords list

stopwords_list += ['--', 'xxxx']

In [ ]:
# Check out word counts

text_words = process_narrative(text)

word_counts = {}
for i in range(len(text_words)):
    word_counts[text_words[i]] = text_words.count(text_words[i])
word_counts

It seems there are a lot of numbers. Update function to get rid of numbers from the list.  

Note: this function also gets rid of strings with punctuation in it like 'xx/xx/xxxx' or "n't".

In [ ]:
def process_narrative(narrative):
    tokens = nltk.word_tokenize(narrative)
    stopwords_removed = [token.lower() for token in tokens if token.lower() not in stopwords_list]
    
    # adding line to remove all tokens with numbers and punctuation
    stopwords_punc_and_numbers_removed = [word for word in stopwords_removed if word.isalpha()]
    
    return stopwords_punc_and_numbers_removed  

In [ ]:
# Redoing processing with updated function
text_words = process_narrative(text)

### Make dictionary of word counts

In [ ]:
FreqDist(text_words)

In [ ]:
type(FreqDist(text_words))

In [ ]:
FreqDist(text_words).most_common(10)

Note how calling `most_common()` creates a list of tuples.

In [ ]:
FreqDist(text_words).plot(10)

### Trying process on the next two rows

#### df.iloc[1]

In [ ]:
df.iloc[5][0:5]

In [ ]:
text = df['narrative'][5]
text

In [ ]:
text_words = process_narrative(text)

In [ ]:
FreqDist(text_words).most_common(30)

#### df.iloc[2]

In [ ]:
df.iloc[8][0:5]

In [ ]:
text = df['narrative'][8]
text_words = process_narrative(text)
FreqDist(text_words).most_common(30)

In [ ]:
df.iloc[2]

# Combine categories and create new dataframes

## Inspect 

In [ ]:
# Inspect products again
df['product'].value_counts()

What is "Money transfer, virtual currency, or money service"?

In [ ]:
df[df['product'] == "Money transfer, virtual currency, or money service"].head(10)

Seems to be about Venmo, digital transactions, international transfers, etc. It's a bit of it's own thing. I'll keep it for now, but there are only 4,602 entries. But I'll fold into "checking and savings" in general, which has only 9,000.

## Combine categories

**Tasks**

- Rename "credit_reporting"  
- Rename "debt_collection"  
- Rename "credit_card"
- Rename "mortgage"
- Combine "checking" and "money transfer" into "retail_banking"
- Combine the loans into "loans"

In [ ]:
df['product'].value_counts()

In [ ]:
df['product'] = df['product'].astype(str).replace({'Credit reporting, credit repair services, or other personal consumer reports': 'credit_reporting',
                       'Debt collection': 'debt_collection',
                       'Credit card or prepaid card': 'credit_card',
                       'Mortgage': 'mortgage',
                       'Checking or savings account': 'retail_banking',
                       'Money transfer, virtual currency, or money service': 'retail_banking',
                       'Vehicle loan or lease': 'loans',
                       'Payday loan, title loan, or personal loan': 'loans',
                       'Student loan': 'loans'}, inplace=True)

In [ ]:
df['product'].unique()


In [ ]:
df['product'].value_counts()

Mortgage and loans are the smallest. Since they're both types of loans, I'll combine them.

In [ ]:
df['product'] = df['product'].replace({'mortgage': 'mortgages_and_loans',
                       'loans': 'mortgages_and_loans'}, inplace=True)
df['product'].value_counts()

In [ ]:
df['product']

## Create new dataframes

In [ ]:
credit_reporting_df = df[df['product'] == 'credit_reporting']
debt_collection_df = df[df['product'] == 'debt_collection']
mortgages_and_loans_df = df[df['product'] == 'mortgages_and_loans']
credit_card_df = df[df['product'] == 'credit_card']
retail_banking_df = df[df['product'] == 'retail_banking']

## Concatenate all the narratives into a single string per class

In [ ]:
credit_reporting_df.head()

In [ ]:
mortgages_and_loans_df.head()

In [ ]:
def concat_narratives(df):
    # concat narratives, drop NaN values
    narr = ''.join(df['narrative'].dropna().astype(str))
    print('Finished Concatenation')
    return narr

In [ ]:
credit_reporting_text = concat_narratives(credit_reporting_df)
credit_reporting_text_processed = process_narrative(credit_reporting_text)

In [ ]:
debt_collection_text = concat_narratives(debt_collection_df)
debt_collection_text_processed = process_narrative(debt_collection_text)

In [ ]:
mortgages_and_loans_text = concat_narratives(mortgages_and_loans_df)
mortgages_and_loans_text_processed = process_narrative(mortgages_and_loans_text)

In [ ]:
credit_card_text = concat_narratives(credit_card_df)
credit_card_text_processed = process_narrative(credit_card_text)

In [ ]:
retail_banking_text = concat_narratives(retail_banking_df)
retail_banking_text_processed = process_narrative(retail_banking_text)

### Saving the text files

In [ ]:
text_file = open('../data/credit_reporting_text.txt', 'w')
text_file.write(credit_reporting_text)
text_file.close()

In [ ]:
text_file = open('../data/debt_collection_text.txt', 'w')
text_file.write(debt_collection_text)
text_file.close()

In [ ]:
text_file = open('../data/mortgages_and_loans_text.txt', 'w')
mortgages_and_loans_text = mortgages_and_loans_text.replace('\x82', '')
text_file.write(mortgages_and_loans_text)
text_file.close()

In [ ]:
text_file = open('../data/credit_card_text.txt', 'w')
text_file.write(credit_card_text)
text_file.close()

In [ ]:
text_file = open('../data/retail_banking_text.txt', 'w')
text_file.write(retail_banking_text)
text_file.close()

### Saving the processed text (lists) files

In [ ]:
temp = pd.DataFrame(credit_reporting_text_processed)
temp.to_csv('../data/credit_reporting_text_processed.csv')

In [ ]:
temp = pd.DataFrame(debt_collection_text_processed)
temp.to_csv('../data/debt_collection_text_processed.csv')

In [ ]:
temp = pd.DataFrame(mortgages_and_loans_text_processed)
temp.to_csv('../data/mortgages_and_loans_text_processed.csv')

In [ ]:
temp = pd.DataFrame(credit_card_text_processed)
temp.to_csv('../data/credit_card_text_processed.csv')

In [ ]:
temp = pd.DataFrame(retail_banking_text_processed)
temp.to_csv('../data/retail_banking_text_processed.csv')

## Check `FreqDist()`

In [ ]:
FreqDist(debt_collection_text_processed).most_common(30)

In [ ]:
FreqDist(credit_reporting_text_processed).most_common(30)

In [ ]:
FreqDist(credit_card_text_processed).most_common(30)

In [ ]:
FreqDist(retail_banking_text_processed).most_common(30)

In [ ]:
FreqDist(mortgages_and_loans_text_processed).most_common(30)